# មេរៀន ០១ - ចំណេះដឹងទូទៅអំពីភ្នាក់ងារបញ្ញាសិប្បនិម្មិត

សូមស្វាគមន៍មកកាន់មេរៀនដំបូងក្នុងវគ្គ **ភ្នាក់ងារបញ្ញាសិប្បនិម្មិតសម្រាប់អ្នកចាប់ផ្តើម**!

**ភ្នាក់ងារបញ្ញាសិប្បនិម្មិត** គឺជាកម្មវិធីមួយដែលប្រើម៉ូដែលភាសា​ធំមួយ (LLM) ជាឧបករណ៍ហេតុផលរបស់វា ហើយអាចធ្វើសកម្មភាពនៅក្នុងពិភពពិត — ការហៅ API, ស្វែងរកទិន្នន័យនៅក្នុងទិន្នន័យសន្ដិសុខ ឬរត់កូដ — ដើម្បីសំរេចគោលដៅមួយចំពោះអ្នកប្រើ។

នៅក្នុងសៀវភៅកំណត់ត្រានេះ អ្នកនឹងបង្កើតភ្នាក់ងារដំបូងរបស់អ្នក៖ **ភ្នាក់ងារធ្វើដំណើរ** ដែលផ្ដល់អនុសាសន៍កន្លែងស្នាក់នៅក្នុងពិភពចូលរួមធ្វើដំណើរ។ ក្នុងការសិក្សារពេលនេះ អ្នកនឹងរៀនពីរបៀប៖

1. ទំនាក់ទំនងទៅម៉ាស៊ីនមេ Azure AI Foundry Agent ដោយប្រើ **Microsoft Agent Framework**។
2. ផ្តល់ឧបករណ៍មួយសម្រាប់ភ្នាក់ងារ — មុខងារ Python សាមញ្ញមួយដែលវាអាចហៅបាន។
3. ដំណើរការភ្នាក់ងារ និងពិនិត្យចម្លើយរបស់វា។
4. បង្ហាញចម្លើយរបស់ភ្នាក់ងារជាកំណត់ផ្សាយបែប token ដោយ token។


## ការតំឡើងត្រៀមប្រើ

មុនពេលបើកការប្រតិបត្តិនេះ សូមប្រាកដថាអ្នកមាន៖

1. **គម្រោង Azure AI Foundry** រួចជាមួយម៉ូឌែលសន្ទនាកំពុងដំណើរការ (ឧ. `gpt-4o-mini`)។
2. **បានចូលគណនីដោយប្រើ Azure CLI** — ដំណើរការ `az login` នៅក្នុងטרמינលរបស់អ្នក។
3. **បានកំណត់អថេរបរិបទដែលត្រូវការ៖**
   - `AZURE_AI_PROJECT_ENDPOINT` — អាសយដ្ឋានគម្រោង Azure AI Foundry របស់អ្នក។
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — ឈ្មោះម៉ូឌែលដែលបានដាក់ចេញរបស់អ្នក។

កូដខាងក្រោមត្រូវបានដំឡើង​បណ្ណាល័យ Python ដែលអ្នកត្រូវការ។


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## បង្កើតភ្នាក់ងារដំបូងរបស់អ្នក

ភ្នាក់ងារត្រូវការពីរប្រាក់ចាំបាច់ៈ

- **សេចក្ដីណែនាំ** ដែលប្រាប់វា *អ្នកគឺជាអ្វី* និង *របៀបអនុវត្តខ្លួន* (ការលើកទឹកចិត្តប្រព័ន្ធ)។
- **ឧបករណ៍** — មុខងារ Python ដែលបានបង្ហាក់ជាមួយ `@tool` ដែលភ្នាក់ងារអាចហៅដើម្បីទទួលព័ត៌មានឬអនុវត្តសកម្មភាព។

ខាងក្រោមនេះយើងកំណត់ឧបករណ៍សាមញ្ញមួយដែលត្រឡប់បញ្ជីកន្លែងទេសចរណ៍ពេញនិយម។ ភ្នាក់ងារនឹងប្រើឧបករណ៍នេះពេលអ្នកប្រើសួរអំពីការណែនាំដំណើរកំសាន្ត។


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## ចម្លើយចាក់បញ្ចាំង

សម្រាប់បទពិសោធន៍មានអន្តរកម្មច្រើនជាងនេះ អ្នកអាច **ចាក់បញ្ចាំង** ចម្លើយរបស់ភ្នាក់ងារ។ ជំនួសការរង់ចាំចម្លើយពេញលេញ ភ្នាក់ងារនឹងបង្ហាញអត្ថបទជាផ្នែកៗនៅពេលដែលវាត្រូវបានបង្កើត។ របៀបនេះមានប្រយោជន៍ជាពិសេសនៅក្នុងចំណុចជជែក ដែលអ្នកចង់បង្ហាញចេញលទ្ធផលនៅពេលដំណើរការពិត។


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## សេចក្ដីសង្ខេប

នៅក្នុងមេរៀននេះអ្នកបានរៀនពីរបៀប:

- **បង្កើតអ្នកផ្គត់ផ្គង់** ដែលភ្ជាប់ទៅកាន់សេវាកម្ម Azure AI Foundry Agent តាមរយៈ `AzureAIProjectAgentProvider`។
- **កំណត់ឧបករណ៍** ដោយប្រើអ្នកតម្លើង `@tool` ដើម្បីឲ្យភ្នាក់ងារអាចហៅមុខងារ Python របស់អ្នកបាន។
- **បើកដំណើរការភ្នាក់ងារ** ជាមួយសារអ្នកប្រើ និងបោះពុម្ពចម្លើយរបស់វា។
- **ចាក់សោតចម្លើយ** សម្រាប់លទ្ធផលបន្តផ្ទាល់។

នៅក្នុងមេរៀនបន្ទាប់ យើងនឹងស្វែងយល់អំពីស៊ុមភ្រាំង agentic ជ្រាលជ្រៅជាងនេះ និងរៀនពីរបៀបផ្ដល់ឧបករណ៍មានអំណាចជាច្រើន និងសមត្ថភាពហិរញ្ញវត្ថុច្រើនជំហានឲ្យភ្នាក់ងារ។


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
